In [1]:
from collections import deque, defaultdict

class RollingMaxPerUser:
    def __init__(self, window_ms=5*60*1000):
        self.window_ms = window_ms
        self.deques = defaultdict(deque) # userid -> deque((ts,score))
        self.latest_ts = {}

    def process(self,user_id, ts, score):
        self.latest_ts[user_id] = max(self.latest_ts.get(user_id, 0), ts)
        cutoff = self.latest_ts[user_id] - self.window_ms
        dq = self.deques[user_id]

        while dq and dq[0][0] < cutoff: # stale entries
            dq.popleft()

        while dq and dq[-1][1] <= score: # remove smaller scores
            dq.pop()

        dq.append((ts,score))

        return dq[0][1] # max score in last window
        

In [4]:
def sorted_difference(a, b):
    i=j=0
    result = []
    while i < len(a):
        if j >= len(b) or a[i] < b[j]:
            result.append(a[i])
            i+=1
        elif a[i] > b[j]:
            j+=1
        else:
            i+=1
            j+=1
    return result

print(sorted_difference([1,2,3,4,5], [2,4,6,8,10]))

[1, 3, 5]


In [5]:
def multiply_sparse(A, B):
    m,k,n = len(A), len(B), len(B[0])
    sparse_A = [[(j,val) for j,val in enumerate(row) if val != 0] for row in A]

    result = [[0]*n for _ in range(m)]

    for i, row_entries in enumerate(sparse_A):
        for col_A, val_A in row_entries:

            for col_B, val_B in enumerate(B[col_A]):
                if val_B != 0:
                    result[i][col_B] += val_A * val_B
    return result

A = [
    [1, 0, 0],
    [-1, 0, 3]
]

# Matrix B (3x2) - Mostly zeros
B = [
    [7, 0],
    [0, 0],
    [0, 1]
]

# Multiply A x B
result = multiply_sparse(A, B)

# Print the result row by row
for row in result:
    print(row)

[7, 0]
[-7, 3]


In [ ]:
def min_tool_calls_to_goal(duration, target):
    # 'n' is the total number of steps we have available
    n = len(duration)
    INF = float('inf')

    # dp[i][r] answers the question: "Starting at step 'i', if I need 'r' more 
    # progress, what is the absolute minimum number of tool calls required?"
    # We initialize everything to INF (infinity) because we assume it's impossible at first.
    dp = [[INF]*(target + 1) for _ in range(n+1)]
    
    # Base case: We are out of steps (index n) and need 0 more progress.
    # It takes exactly 0 extra tool calls to achieve 0 progress.
    dp[n][0] = 0 

    # We loop backward from the last step (n-1) down to the first step (0).
    # We work backward because the decision we make at step 'i' depends on 
    # knowing the optimal future costs, which we calculate first.
    for i in range(n-1, -1, -1):
        row = dp[i]           # The answers for our current step
        next_row = dp[i+1]    # The answers for the immediate next step
        
        # If we need 0 progress at this current step 'i', we are already done. 
        # It takes 0 calls to finish.
        row[0] = 0

        # 'd' is the amount of progress the tool at this step provides
        d = duration[i]
        
        # Pre-calculate where we jump if we decide to use the tool. 
        # Because the tool takes time, we are forced to skip ahead 'd' steps.
        if d > 0:
            # We jump to i + d, but we cap it at 'n' so we don't go out of bounds
            jump_idx = min(i + d, n)
            jump_row = dp[jump_idx]
        
        # Now, figure out the best move for every possible remaining progress requirement
        for r in range(1, target + 1):
            
            # CHOICE A: Skip the tool at step 'i'.
            # We don't spend a call, so our total cost is just whatever it costs 
            # to get the exact same remaining target 'r' starting from the next step.
            best = next_row[r] 

            # CHOICE B: Use the tool at step 'i' (if it actually gives progress).
            if d > 0:
                # We gain 'd' progress. Our remaining target 'r' shrinks.
                # max(0, r - d) ensures our required progress stops at 0 and doesn't go negative.
                new_r = max(0, r - d) 
                
                # Look up the future cost from our new landing spot ('jump_idx') 
                # needing our new remaining target ('new_r').
                future_calls = jump_row[new_r] 
                
                # If it's actually possible to finish from that future spot (not INF),
                # calculate the total cost: 1 (for using THIS tool) + future_calls.
                # If this total is strictly fewer calls than Choice A, it becomes our new 'best'.
                if future_calls != INF and 1 + future_calls < best:
                    best = 1 + future_calls
            
            # Save the absolute best choice into our DP table for this specific state.
            row[r] = best
            
    # The final answer we want is sitting at the very beginning: 
    # Starting at step 0, needing the full 'target' amount of progress.
    ans = dp[0][target]
    
    # If the answer is still INF, it means we couldn't find any valid combination 
    # of tools to reach the target, so we return -1.
    return ans if ans != INF else -1


min_tool_calls_to_goal([3, 0, 0, 0, 3, 1, 0], 4)

2

In [8]:
def min_tool_calls_to_goal(duration, target):
    n = len(duration)
    INF = float('inf')

    dp = [[INF]*(target+1) for _ in range(n+1)]

    dp[n][0] = 0

    for i in range(n-1,-1,-1):
        row = dp[i]
        next_row = dp[i+1]

        row[0] = 0 # target progress at 0 is 0
        d = duration[i]

        if d > 0:
            jump_idx = min(i+d, n)
            jump_row = dp[jump_idx]

        for r in range(1, target+1):
            #choice A: skip the tool at step i
            best = next_row[r]

            #choice B: do the tool call at step i
            if d > 0:
                new_r = max(0, r-d)
                future_calls = jump_row[new_r] # from the jump row we need only steps to reach new_r

                # if 1 + future_calls (choice B) < best (choice A skipping)
                if future_calls != INF and 1 + future_calls < best:
                    best = 1+ future_calls

            row[r] = best

        ans = dp[0][target]
        return ans if ans != INF else -1

In [9]:
def min_bandwidth(transfers, T):
    n = len(transfers)
    if T < n:
        return -1
    
    def total_time(B):
        s = 0
        for t in transfers:
            s += (t + B - 1)//B # ceil(x/B)
            if s > T: #early exit
                return s
        return s

    lo, hi = 1, max(transfers)
    while lo < hi:
        mid = (lo+hi)//2
        if total_time(mid) <= T:
            hi = mid
        else:
            lo = mid + 1
    return lo

In [ ]:
import numpy as np

class KMeans:
    def __init__(self, k, max_iters, tol, random_state):
        self.k = k
        self.max_iters = max_iters
        self.tol = tol
        self.random_state = random_state
        self.centroids = None
        self.labels = None

    def fit(self, X):
        X = np.asarray(X, dtype=float)
        n_samples = X.shape[0]
        if self.k > n_samples:
            raise
        rng = np.random.default_rng(self.random_state)
        indices = np.random.choice(n_samples, self.k, replace=False)
        self.centroids = X[indices].copy()

        for _ in range(self.max_iters):
            self.labels = self._assign_labels(X)
            new_centroids = self._update_centroids(X, self.labels, rng)
            shift = np.linalg.norm(new_centroids - self.centroids)
            if shift <= self.tol:
                break
        self.labels = self._assign_labels(X)
        return self


    def _assign_labels(self, X):
        squared_dist = np.sum((X[:, None, :] - self.centroids[None, :, :])**2, axis=2)
        return np.argmin(squared_dist, axis=1)

    def _update_centroids(self, X, labels, rng):
        new_centroids = np.empty((self.k, X.shape[1]))
        for cluster_id in range(self.k):
            members = X[labels == cluster_id]
            if len(members) > 0:
                new_centroids[cluster_id] = members.mean(axis = 0)
            else:
                distances = np.min(np.sum((X[:,None,:] - self.centroids[None,:,:])**2, axis = 2), axis=1)
                new_centroids[cluster_id] = X[np.argmax(distances)]
        return new_centroids

    def predict(self, X):
        return self._assign_labels(np.asarray(X, dtype=float))
                



In [ ]:
def solve_n_queens(n: int):
    solutions = []
    board = [["."] * n for _ in range(n)]
    
    # Lookup sets for constant-time constraint checks
    cols = set()         # Tracks occupied columns
    main_diag = set()    # Tracks occupied main diagonals (row - col)
    anti_diag = set()    # Tracks occupied anti-diagonals (row + col)
    
    def backtrack(row):
        # Base Case: All queens successfully placed
        if row == n:
            solutions.append(["".join(r) for r in board])
            return
        
        for col in range(n):
            # Calculate diagonal signatures
            md_id = row - col
            ad_id = row + col
            
            # If the position is under attack, skip it
            if col in cols or md_id in main_diag or ad_id in anti_diag:
                continue
                
            # Place the queen and mark its territory
            board[row][col] = "Q"
            cols.add(col)
            main_diag.add(md_id)
            anti_diag.add(ad_id)
            
            # Recurse into the next row
            backtrack(row + 1)
            
            # Backtrack: Remove the queen and clear the markers
            board[row][col] = "."
            cols.remove(col)
            main_diag.remove(md_id)
            anti_diag.remove(ad_id)

    backtrack(0)
    return solutions

# Example Usage:
# print(solve_n_queens(4))


array([2, 6, 1])

In [14]:
X = np.array([[1,2],[2,3],[3,5],[5,4]])

In [15]:
indices = np.random.choice(4, 3, replace=False)
centroids = X[indices].copy()

In [16]:
centroids

array([[1, 2],
       [3, 5],
       [2, 3]])

In [23]:
np.argmin(np.sum((X[:,None,:] - centroids[None,:,:])**2,axis=2), axis=1)

array([0, 2, 1, 1])